# GestuRhythm - Seq2Seq Music Generation
**Bai toan:** 30 frame cu chi -> chuoi 16 not co cau truc
**Model:** Transformer Encoder-Decoder (kieu GPT)
**Output:** gesture_seq2seq.pt

In [ ]:
# Kiem tra GPU
import torch, os
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    # T4x2: dung DataParallel
    N_GPU = torch.cuda.device_count()
    print(f'So GPU: {N_GPU}')


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

X = np.load('/kaggle/input/gesturhythm/sequences.npy')   # (N, 30, 126)
y = np.load('/kaggle/input/gesturhythm/labels.npy')       # (N, 6)
print(f'X: {X.shape} | y: {y.shape}')

PITCH_MIN, PITCH_MAX = 60, 72
VEL_MIN,   VEL_MAX   = 40, 127
TEMPO_MIN, TEMPO_MAX = 80, 160
SEQ_OUT   = 16   # so not output
NOTE_DIM  =  3   # pitch_norm, velocity_norm, duration_norm

# Scale theo chord type
CHORD_SCALES = {
    0: [0,2,4,5,7,9,11],   # major
    1: [0,2,4,5,7,9,11],   # major
    2: [0,2,3,5,7,8,10],   # minor
    3: [0,2,4,5,7,9,10],   # dominant
}

## 1. Sinh chuoi not tu labels (Music Theory Engine)

In [ ]:
def generate_note_sequence(pitch, velocity, tempo, chord, seq_len=SEQ_OUT):
    """
    Tu 1 label (pitch, velocity, tempo, chord) sinh chuoi seq_len not.
    Ap dung nhac ly: scale snap, rhythm quantize, contour.
    """
    scale_intervals = CHORD_SCALES[int(chord)]
    root = 60  # C4
    scale_notes = [root + i + 12*o for o in range(2) for i in scale_intervals]
    scale_notes = sorted(set(n for n in scale_notes if PITCH_MIN <= n <= PITCH_MAX + 12))

    # Tim vi tri pitch trong scale
    base_idx = min(range(len(scale_notes)), key=lambda i: abs(scale_notes[i] - pitch))

    # Rhythm: tempo -> note duration
    beat_dur = 60.0 / max(tempo, 60)  # giay/beat
    # Toc do cao = nhieu not ngan, toc do thap = it not dai
    if tempo > 130:
        durations = [beat_dur * 0.5] * seq_len           # 1/8 note
    elif tempo > 100:
        durations = [beat_dur * d for d in ([0.5,0.5,1.0,0.5] * (seq_len//4 + 1))[:seq_len]]
    else:
        durations = [beat_dur * 1.0] * seq_len           # 1/4 note

    # Contour: tao melodic shape tu pitch (len/xuong/vong)
    contour_patterns = {
        'ascending':  [0,1,2,3,4,3,2,1,2,3,4,5,4,3,2,1],
        'descending': [5,4,3,2,1,2,3,2,1,0,1,2,1,0,1,2],
        'arch':       [0,1,2,3,4,5,4,3,2,1,2,3,4,3,2,1],
        'wave':       [0,2,1,3,2,4,3,2,1,3,2,1,3,2,4,3],
    }
    # Chon contour theo velocity
    if velocity > 100:   pattern = contour_patterns['ascending']
    elif velocity > 75:  pattern = contour_patterns['arch']
    elif velocity > 50:  pattern = contour_patterns['wave']
    else:                pattern = contour_patterns['descending']

    notes = []
    vel_variation = np.random.normal(0, 5, seq_len).astype(int)
    for i in range(seq_len):
        idx = (base_idx + pattern[i % len(pattern)]) % len(scale_notes)
        p   = scale_notes[idx]
        v   = int(np.clip(velocity + vel_variation[i], VEL_MIN, VEL_MAX))
        d   = durations[i % len(durations)]
        # Normalize ve [0,1]
        notes.append([
            (p - PITCH_MIN) / (PITCH_MAX - PITCH_MIN),
            (v - VEL_MIN)   / (VEL_MAX - VEL_MIN),
            min(d / 2.0, 1.0)
        ])
    return np.array(notes, dtype=np.float32)  # (seq_len, 3)

# Test
test_seq = generate_note_sequence(65, 80, 120, 1)
print(f'Note sequence shape: {test_seq.shape}')
print(f'Sample notes (pitch_norm, vel_norm, dur_norm):')
print(test_seq[:4])

In [ ]:
# Sinh note sequences cho toan bo dataset
print('Sinh note sequences...')
note_sequences = []
for i in range(len(y)):
    pitch, velocity, tempo, chord = int(y[i,0]), int(y[i,1]), int(y[i,2]), int(y[i,3])
    seq = generate_note_sequence(pitch, velocity, tempo, chord)
    note_sequences.append(seq)

note_sequences = np.array(note_sequences)  # (N, 16, 3)
print(f'Note sequences: {note_sequences.shape}')

# Bieu do phan bo not sinh ra
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
fig.suptitle('Generated Note Sequence Distribution', fontweight='bold')
titles = ['Pitch (normalized)', 'Velocity (normalized)', 'Duration (normalized)']
for i, title in enumerate(titles):
    axes[i].hist(note_sequences[:,:,i].flatten(), bins=30, color='steelblue', edgecolor='black')
    axes[i].set_title(title)
    axes[i].set_ylabel('Count')
plt.tight_layout()
plt.savefig('note_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

## 2. Dataset (Teacher Forcing)

In [ ]:
class GestureSeq2SeqDataset(Dataset):
    def __init__(self, X, note_seqs):
        self.X   = torch.tensor(X,         dtype=torch.float32)
        self.tgt = torch.tensor(note_seqs, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        tgt_in     = torch.zeros(SEQ_OUT, NOTE_DIM)
        tgt_in[1:] = self.tgt[idx][:-1]
        return self.X[idx], tgt_in, self.tgt[idx]

dataset  = GestureSeq2SeqDataset(X, note_sequences)
n_train  = int(0.8 * len(dataset))
n_val    = len(dataset) - n_train
train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=32)
print(f'Train: {n_train} | Val: {n_val}')


## 3. Model: Causal Encoder + Autoregressive Decoder

In [ ]:
class GestureEncoder(nn.Module):
    def __init__(self, input_dim=126, d_model=128, nhead=4, num_layers=3):
        super().__init__()
        self.embed   = nn.Linear(input_dim, d_model)
        self.pos_enc = nn.Embedding(100, d_model)
        enc_layer    = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=256,
            dropout=0.1, batch_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

    def forward(self, x):
        B, T, _ = x.shape
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        pos  = torch.arange(T, device=x.device).unsqueeze(0)
        x    = self.embed(x) + self.pos_enc(pos)
        return self.transformer(x, mask=mask)

class MusicDecoder(nn.Module):
    def __init__(self, note_dim=NOTE_DIM, d_model=128, nhead=4, num_layers=3):
        super().__init__()
        self.note_embed = nn.Linear(note_dim, d_model)
        self.pos_enc    = nn.Embedding(100, d_model)
        dec_layer       = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=256,
            dropout=0.1, batch_first=True)
        self.transformer = nn.TransformerDecoder(dec_layer, num_layers=num_layers)
        self.out_proj    = nn.Linear(d_model, note_dim)

    def forward(self, tgt, memory):
        B, T, _ = tgt.shape
        tgt_mask = torch.triu(torch.ones(T, T, device=tgt.device), diagonal=1).bool()
        pos  = torch.arange(T, device=tgt.device).unsqueeze(0)
        tgt  = self.note_embed(tgt) + self.pos_enc(pos)
        return self.out_proj(self.transformer(tgt, memory, tgt_mask=tgt_mask))

class GestureSeq2Seq(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = GestureEncoder()
        self.decoder = MusicDecoder()

    def forward(self, src, tgt):
        return self.decoder(tgt, self.encoder(src))

    def generate(self, src, seq_len=SEQ_OUT):
        self.eval()
        with torch.no_grad():
            memory    = self.encoder(src)
            generated = torch.zeros(src.shape[0], 1, NOTE_DIM, device=src.device)
            for _ in range(seq_len):
                out       = self.decoder(generated, memory)
                generated = torch.cat([generated, out[:, -1:, :]], dim=1)
            return generated[:, 1:, :]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = GestureSeq2Seq().to(device)
total  = sum(p.numel() for p in model.parameters())
print(f'Device: {device}')
print(f'Encoder: {sum(p.numel() for p in model.encoder.parameters()):,} params')
print(f'Decoder: {sum(p.numel() for p in model.decoder.parameters()):,} params')
print(f'Total  : {total:,} params')


## 4. Training

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-3,
    steps_per_epoch=len(train_loader), epochs=100)
criterion = nn.MSELoss()

EPOCHS = 100
train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    model.train()
    t_loss = 0
    for src, tgt_in, tgt_out in train_loader:
        src, tgt_in, tgt_out = src.to(device), tgt_in.to(device), tgt_out.to(device)
        optimizer.zero_grad()
        loss = criterion(model(src, tgt_in), tgt_out)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        t_loss += loss.item()
    model.eval()
    v_loss = 0
    with torch.no_grad():
        for src, tgt_in, tgt_out in val_loader:
            src, tgt_in, tgt_out = src.to(device), tgt_in.to(device), tgt_out.to(device)
            v_loss += criterion(model(src, tgt_in), tgt_out).item()
    train_losses.append(t_loss / len(train_loader))
    val_losses.append(v_loss  / len(val_loader))
    if (epoch+1) % 20 == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | train={train_losses[-1]:.4f} | val={val_losses[-1]:.4f}')
print('Done!')


## 5. Visualization & Evaluation

In [ ]:
# Loss curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('GestuRhythm Seq2Seq - Training Results', fontweight='bold')

axes[0].plot(train_losses, label='Train', color='steelblue')
axes[0].plot(val_losses,   label='Val',   color='coral')
axes[0].set_title('Loss Curve'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Sample generation - lay 1 sample tu val set
model.eval()
sample_src, _, sample_tgt = next(iter(val_loader))
generated = model.generate(sample_src[:1].to(device)).cpu().numpy()[0]  # (16, 3)
true_notes = sample_tgt[0].numpy()

# Denormalize pitch
gen_pitch  = generated[:,0]  * (PITCH_MAX - PITCH_MIN) + PITCH_MIN
true_pitch = true_notes[:,0] * (PITCH_MAX - PITCH_MIN) + PITCH_MIN
gen_vel    = generated[:,1]  * (VEL_MAX - VEL_MIN) + VEL_MIN
true_vel   = true_notes[:,1] * (VEL_MAX - VEL_MIN) + VEL_MIN

x = range(SEQ_OUT)
axes[1].step(x, gen_pitch,  where='post', label='Generated', color='steelblue', linewidth=2)
axes[1].step(x, true_pitch, where='post', label='Target',    color='coral',     linewidth=2, linestyle='--')
axes[1].set_title('Sample: Pitch Sequence'); axes[1].set_xlabel('Note index')
axes[1].set_ylabel('MIDI pitch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].step(x, gen_vel,  where='post', label='Generated', color='steelblue', linewidth=2)
axes[2].step(x, true_vel, where='post', label='Target',    color='coral',     linewidth=2, linestyle='--')
axes[2].set_title('Sample: Velocity Sequence'); axes[2].set_xlabel('Note index')
axes[2].set_ylabel('MIDI velocity'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('seq2seq_results.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Attention Visualization (Encoder)

In [ ]:
# Trich xuat attention weights tu encoder layer dau tien
hooks, attn_weights = [], []

def hook_fn(module, input, output):
    # TransformerEncoderLayer tra ve (output, attn_weights) neu need_weights=True
    pass

# Lay attention bang cach goi truc tiep
src_sample = sample_src[:1].to(device)
with torch.no_grad():
    enc_out = model.encoder.embed(src_sample)
    pos = torch.arange(30, device=device).unsqueeze(0)
    enc_out = enc_out + model.encoder.pos_enc(pos)
    # Layer 0 attention
    layer0 = model.encoder.transformer.layers[0]
    attn_out, attn_w = layer0.self_attn(enc_out, enc_out, enc_out, need_weights=True)
    attn_w = attn_w.squeeze().cpu().numpy()  # (30, 30)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(attn_w, cmap='Blues', aspect='auto')
ax.set_title('Encoder Self-Attention Weights (Layer 0)\n'
             'Moi o (i,j): Frame i chu y den Frame j bao nhieu', fontsize=11)
ax.set_xlabel('Source Frame (Key)')
ax.set_ylabel('Query Frame')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig('attention_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('Attention map: hang i = frame i dang chu y vao frame nao nhat')


## 7. Save Model

In [ ]:
torch.save({
    'model_state': model.state_dict(),
    'config': {
        'input_dim': 126, 'd_model': 128, 'nhead': 4,
        'num_layers': 3,  'note_dim': NOTE_DIM, 'seq_out': SEQ_OUT
    },
    'label_config': {
        'pitch_min': PITCH_MIN, 'pitch_max': PITCH_MAX,
        'vel_min': VEL_MIN,     'vel_max': VEL_MAX,
        'tempo_min': TEMPO_MIN, 'tempo_max': TEMPO_MAX,
        'chord_scales': CHORD_SCALES
    },
    'train_losses': train_losses,
    'val_losses':   val_losses,
}, 'gesture_seq2seq.pt')
print('Saved: gesture_seq2seq.pt')
print(f'Final val loss: {val_losses[-1]:.4f}')


## 8. Phan tich Dataset Chi Tiet

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

fig = plt.figure(figsize=(16, 10))
fig.suptitle('GestuRhythm - Dataset Analysis', fontsize=15, fontweight='bold')
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.4, wspace=0.35)

# 1. Phan bo pitch
ax1 = fig.add_subplot(gs[0,0])
pitch_map = {60:'C4',62:'D4',65:'F4',69:'A4',72:'C5'}
pitches, counts = np.unique(y[:,0], return_counts=True)
bars = ax1.bar([pitch_map.get(int(p),str(int(p))) for p in pitches], counts,
               color=['#2ecc71','#3498db','#e74c3c','#f39c12','#9b59b6'], edgecolor='black')
ax1.set_title('Pitch Distribution'); ax1.set_ylabel('Samples')
for bar, v in zip(bars, counts): ax1.text(bar.get_x()+bar.get_width()/2, v+0.5, str(v), ha='center', fontsize=8)

# 2. Chord distribution
ax2 = fig.add_subplot(gs[0,1])
chord_map = {0:'None',1:'Major',2:'Minor',3:'Dom'}
chords, ccounts = np.unique(y[:,3], return_counts=True)
colors_c = ['#95a5a6','#2ecc71','#e74c3c','#3498db']
bars2 = ax2.bar([chord_map[int(c)] for c in chords], ccounts,
                color=[colors_c[int(c)] for c in chords], edgecolor='black')
ax2.set_title('Chord Distribution'); ax2.set_ylabel('Samples')
for bar, v in zip(bars2, ccounts): ax2.text(bar.get_x()+bar.get_width()/2, v+0.5, str(v), ha='center', fontsize=8)

# 3. Velocity histogram
ax3 = fig.add_subplot(gs[0,2])
ax3.hist(y[:,1], bins=25, color='coral', edgecolor='black', alpha=0.8)
ax3.axvline(y[:,1].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean={y[:,1].mean():.0f}')
ax3.set_title('Velocity Distribution'); ax3.set_xlabel('Velocity'); ax3.legend()

# 4. Tempo histogram
ax4 = fig.add_subplot(gs[0,3])
ax4.hist(y[:,2], bins=25, color='mediumpurple', edgecolor='black', alpha=0.8)
ax4.axvline(y[:,2].mean(), color='purple', linestyle='--', linewidth=2, label=f'Mean={y[:,2].mean():.0f}')
ax4.set_title('Tempo Distribution'); ax4.set_xlabel('BPM'); ax4.legend()

# 5. Hand presence
ax5 = fig.add_subplot(gs[1,0])
rp, lp = y[:,4], y[:,5]
hcounts = [((rp==1)&(lp==0)).sum(),((rp==0)&(lp==1)).sum(),
           ((rp==1)&(lp==1)).sum(),((rp==0)&(lp==0)).sum()]
hlabels = ['Only\nRight','Only\nLeft','Both','None']
hcolors = ['#f39c12','#1abc9c','#9b59b6','#95a5a6']
bars5 = ax5.bar(hlabels, hcounts, color=hcolors, edgecolor='black')
ax5.set_title('Hand Presence')
for bar, v in zip(bars5, hcounts): ax5.text(bar.get_x()+bar.get_width()/2, v+0.5, str(v), ha='center', fontsize=8)

# 6. Sample gesture sequence (wrist xy)
ax6 = fig.add_subplot(gs[1,1])
sample = X[10]  # (30, 126)
ax6.plot(sample[:,0], label='wrist x', color='steelblue')
ax6.plot(sample[:,1], label='wrist y', color='coral')
ax6.set_title('Sample Gesture (Right Wrist)')
ax6.set_xlabel('Frame'); ax6.legend(); ax6.grid(True, alpha=0.3)

# 7. Note sequence example
ax7 = fig.add_subplot(gs[1,2])
ex_notes = note_sequences[10]  # (16, 3)
pitch_vals = ex_notes[:,0] * (PITCH_MAX-PITCH_MIN) + PITCH_MIN
ax7.step(range(SEQ_OUT), pitch_vals, where='post', color='#2ecc71', linewidth=2)
ax7.fill_between(range(SEQ_OUT), pitch_vals, step='post', alpha=0.3, color='#2ecc71')
ax7.set_title('Sample Note Sequence (Pitch)')
ax7.set_xlabel('Note index'); ax7.set_ylabel('MIDI Pitch'); ax7.grid(True, alpha=0.3)

# 8. Pitch vs Tempo scatter
ax8 = fig.add_subplot(gs[1,3])
sc = ax8.scatter(y[:,2], y[:,0], c=y[:,3], cmap='RdYlGn', alpha=0.5, s=15)
plt.colorbar(sc, ax=ax8, label='Chord')
ax8.set_title('Pitch vs Tempo (color=Chord)')
ax8.set_xlabel('Tempo (BPM)'); ax8.set_ylabel('MIDI Pitch')

plt.savefig('dataset_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: dataset_analysis.png')


## 9. Danh gia Model sau Training

In [ ]:
# MAE tren val set
model.eval()
all_gen, all_tgt = [], []
with torch.no_grad():
    for src, tgt_in, tgt_out in val_loader:
        gen = model.generate(src.to(device)).cpu().numpy()
        all_gen.append(gen)
        all_tgt.append(tgt_out.numpy())

all_gen = np.vstack(all_gen)  # (N_val, 16, 3)
all_tgt = np.vstack(all_tgt)

# Denormalize
gen_p = all_gen[:,:,0] * (PITCH_MAX-PITCH_MIN) + PITCH_MIN
tgt_p = all_tgt[:,:,0] * (PITCH_MAX-PITCH_MIN) + PITCH_MIN
gen_v = all_gen[:,:,1] * (VEL_MAX-VEL_MIN) + VEL_MIN
tgt_v = all_tgt[:,:,1] * (VEL_MAX-VEL_MIN) + VEL_MIN
gen_d = all_gen[:,:,2]
tgt_d = all_tgt[:,:,2]

mae_p = np.abs(gen_p - tgt_p).mean()
mae_v = np.abs(gen_v - tgt_v).mean()
mae_d = np.abs(gen_d - tgt_d).mean()

print('=== FINAL EVALUATION ===')
print(f'MAE Pitch    : {mae_p:.2f} semitones')
print(f'MAE Velocity : {mae_v:.2f} / 127')
print(f'MAE Duration : {mae_d:.3f} (normalized)')
print(f'Final Val Loss: {val_losses[-1]:.4f}')


In [ ]:
# Bieu do danh gia toan dien
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('GestuRhythm Seq2Seq - Evaluation', fontsize=14, fontweight='bold')

# 1. Loss curve
axes[0,0].plot(train_losses, label='Train', color='steelblue', linewidth=2)
axes[0,0].plot(val_losses,   label='Val',   color='coral',     linewidth=2)
axes[0,0].set_title('Training Loss Curve')
axes[0,0].set_xlabel('Epoch'); axes[0,0].set_ylabel('MSE Loss')
axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)
axes[0,0].annotate(f'Final val: {val_losses[-1]:.4f}',
    xy=(len(val_losses)-1, val_losses[-1]),
    xytext=(-40, 20), textcoords='offset points',
    arrowprops=dict(arrowstyle='->', color='red'), fontsize=9, color='red')

# 2. Pred vs True pitch (scatter)
axes[0,1].scatter(tgt_p.flatten(), gen_p.flatten(), alpha=0.3, s=5, color='steelblue')
axes[0,1].plot([PITCH_MIN,PITCH_MAX],[PITCH_MIN,PITCH_MAX],'r--', linewidth=2, label='Perfect')
axes[0,1].set_title('Pitch: Predicted vs True')
axes[0,1].set_xlabel('True'); axes[0,1].set_ylabel('Predicted')
axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

# 3. Pred vs True velocity
axes[0,2].scatter(tgt_v.flatten(), gen_v.flatten(), alpha=0.3, s=5, color='coral')
axes[0,2].plot([VEL_MIN,VEL_MAX],[VEL_MIN,VEL_MAX],'r--', linewidth=2, label='Perfect')
axes[0,2].set_title('Velocity: Predicted vs True')
axes[0,2].set_xlabel('True'); axes[0,2].set_ylabel('Predicted')
axes[0,2].legend(); axes[0,2].grid(True, alpha=0.3)

# 4. Generated vs Target - 1 sample piano roll
ax = axes[1,0]
idx = 0
g_p = all_gen[idx,:,0] * (PITCH_MAX-PITCH_MIN) + PITCH_MIN
t_p = all_tgt[idx,:,0] * (PITCH_MAX-PITCH_MIN) + PITCH_MIN
g_v = all_gen[idx,:,1]
t_v = all_tgt[idx,:,1]
x   = range(SEQ_OUT)
ax.step(x, g_p, where='post', color='steelblue', linewidth=2, label='Generated')
ax.step(x, t_p, where='post', color='coral',     linewidth=2, label='Target', linestyle='--')
ax.set_title('Sample Pitch Sequence'); ax.set_xlabel('Note index')
ax.set_ylabel('MIDI Pitch'); ax.legend(); ax.grid(True, alpha=0.3)

# 5. Piano roll visualization
ax5 = axes[1,1]
for i in range(SEQ_OUT):
    p = all_gen[0,i,0] * (PITCH_MAX-PITCH_MIN) + PITCH_MIN
    v = all_gen[0,i,1]
    d = max(all_gen[0,i,2] * 10, 0.3)
    ax5.barh(p, d, left=i, height=0.8, color=plt.cm.YlOrRd(v), edgecolor='gray', linewidth=0.5)
ax5.set_title('Generated Piano Roll'); ax5.set_xlabel('Time (note index)')
ax5.set_ylabel('MIDI Pitch'); ax5.grid(True, alpha=0.2)

# 6. MAE per note position
mae_per_pos = np.abs(gen_p - tgt_p).mean(axis=0)
axes[1,2].bar(range(SEQ_OUT), mae_per_pos, color='mediumpurple', edgecolor='black')
axes[1,2].set_title('MAE per Note Position')
axes[1,2].set_xlabel('Note index'); axes[1,2].set_ylabel('MAE (semitones)')
axes[1,2].axhline(mae_per_pos.mean(), color='red', linestyle='--', label=f'Mean={mae_per_pos.mean():.2f}')
axes[1,2].legend(); axes[1,2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: evaluation_results.png')


## 10. Attention Visualization

In [ ]:
# Heatmap attention encoder layer 0
src_s = next(iter(val_loader))[0][:1].to(device)
with torch.no_grad():
    e = model.encoder.embed(src_s)
    pos = torch.arange(30, device=device).unsqueeze(0)
    e = e + model.encoder.pos_enc(pos)
    _, attn_w = model.encoder.transformer.layers[0].self_attn(
        e, e, e, need_weights=True, average_attn_weights=True)
    attn_w = attn_w.squeeze().cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Transformer Attention Analysis', fontsize=13, fontweight='bold')

# Encoder attention heatmap
im = axes[0].imshow(attn_w, cmap='Blues', aspect='auto')
axes[0].set_title('Encoder Self-Attention (Layer 0)\n'
                  'O(i,j): Frame i chu y den Frame j')
axes[0].set_xlabel('Key Frame'); axes[0].set_ylabel('Query Frame')
plt.colorbar(im, ax=axes[0])

# Attention weight theo thoi gian (frame nao duoc chu y nhat)
mean_attn = attn_w.mean(axis=0)  # trung binh theo query
axes[1].bar(range(30), mean_attn, color='steelblue', edgecolor='black', alpha=0.8)
axes[1].set_title('Average Attention per Frame\n'
                  'Frame nao quan trong nhat khi predict?')
axes[1].set_xlabel('Frame index (0=dau, 29=cuoi)')
axes[1].set_ylabel('Average attention weight')
axes[1].grid(True, alpha=0.3)
peak = mean_attn.argmax()
axes[1].annotate(f'Peak: frame {peak}',
    xy=(peak, mean_attn[peak]),
    xytext=(10, -20), textcoords='offset points',
    arrowprops=dict(arrowstyle='->', color='red'), color='red', fontsize=10)

plt.tight_layout()
plt.savefig('attention_visualization.png', dpi=150, bbox_inches='tight')
plt.show()


## 11. Music Quality Metrics

In [ ]:
from collections import Counter
import numpy as np

def scale_adherence(pitch_seq, scale_notes):
    """% not nam dung trong scale."""
    pcs = [int(p) % 12 for p in pitch_seq if p > 0]
    scale_pcs = [s % 12 for s in scale_notes]
    return sum(p in scale_pcs for p in pcs) / max(len(pcs), 1)

def pitch_entropy(pitch_seq):
    """Do da dang cua melody (cao = da dang, thap = don dieu)."""
    pcs = [int(p) % 12 for p in pitch_seq if p > 0]
    if not pcs: return 0
    counts = Counter(pcs)
    total  = sum(counts.values())
    probs  = [v/total for v in counts.values()]
    return -sum(p * np.log2(p + 1e-10) for p in probs)

def rhythmic_consistency(dur_seq):
    """Do on dinh nhip dieu (gan 1 = on dinh)."""
    if len(dur_seq) < 2: return 1.0
    diffs = np.abs(np.diff(dur_seq))
    return 1.0 - float(diffs.mean())

C_MAJOR    = [60,62,64,65,67,69,71]
C_MINOR    = [60,62,63,65,67,68,70]
PENTATONIC = [60,62,64,67,69]

# Tinh metrics tren val set
sar_list, ent_list, rc_list = [], [], []
for i in range(len(all_gen)):
    p_seq = all_gen[i,:,0] * (PITCH_MAX - PITCH_MIN) + PITCH_MIN
    d_seq = all_gen[i,:,2]
    sar_list.append(scale_adherence(p_seq, C_MAJOR))
    ent_list.append(pitch_entropy(p_seq))
    rc_list.append(rhythmic_consistency(d_seq))

print('=== MUSIC QUALITY METRICS ===')
print(f'Scale Adherence Rate : {np.mean(sar_list):.3f} (1.0 = 100% dung scale)')
print(f'Pitch Entropy        : {np.mean(ent_list):.3f} (cao = da dang melody)')
print(f'Rhythmic Consistency : {np.mean(rc_list):.3f} (1.0 = nhip hoan toan on dinh)')


## 12. Bieu do Music Quality & Per-Person Analysis

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('GestuRhythm - Music Quality & Dataset Analysis', fontsize=14, fontweight='bold')

# 1. Music metrics bar chart
metrics = ['Scale\nAdherence', 'Pitch\nEntropy\n(norm)', 'Rhythmic\nConsistency']
values  = [np.mean(sar_list), np.mean(ent_list)/4, np.mean(rc_list)]
colors  = ['#2ecc71', '#3498db', '#e74c3c']
bars = axes[0,0].bar(metrics, values, color=colors, edgecolor='black')
axes[0,0].set_ylim(0, 1.1)
axes[0,0].set_title('Music Quality Metrics')
axes[0,0].axhline(0.8, color='gray', linestyle='--', alpha=0.5, label='Target 0.8')
axes[0,0].legend()
for bar, v in zip(bars, values):
    axes[0,0].text(bar.get_x()+bar.get_width()/2, v+0.02, f'{v:.2f}', ha='center', fontsize=10)

# 2. Scale adherence distribution
axes[0,1].hist(sar_list, bins=20, color='#2ecc71', edgecolor='black', alpha=0.8)
axes[0,1].axvline(np.mean(sar_list), color='red', linestyle='--', linewidth=2,
                  label=f'Mean={np.mean(sar_list):.2f}')
axes[0,1].set_title('Scale Adherence Distribution')
axes[0,1].set_xlabel('Scale Adherence Rate'); axes[0,1].legend()

# 3. Pitch entropy distribution  
axes[0,2].hist(ent_list, bins=20, color='#3498db', edgecolor='black', alpha=0.8)
axes[0,2].axvline(np.mean(ent_list), color='red', linestyle='--', linewidth=2,
                  label=f'Mean={np.mean(ent_list):.2f}')
axes[0,2].set_title('Pitch Entropy Distribution')
axes[0,2].set_xlabel('Entropy (bits)'); axes[0,2].legend()

# 4. Generated melody diversity - pitch class histogram
all_pitches = (all_gen[:,:,0] * (PITCH_MAX-PITCH_MIN) + PITCH_MIN).flatten()
pc_counts = Counter([int(p)%12 for p in all_pitches if p > 0])
note_names = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
axes[1,0].bar(range(12), [pc_counts.get(i,0) for i in range(12)],
              color='mediumpurple', edgecolor='black')
axes[1,0].set_xticks(range(12)); axes[1,0].set_xticklabels(note_names)
axes[1,0].set_title('Pitch Class Histogram (Generated)')
axes[1,0].set_ylabel('Count')

# 5. Loss curve (final)
axes[1,1].plot(train_losses, color='steelblue', linewidth=2, label='Train')
axes[1,1].plot(val_losses,   color='coral',     linewidth=2, label='Val')
axes[1,1].set_title('Training Loss'); axes[1,1].set_xlabel('Epoch')
axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)
axes[1,1].set_yscale('log')

# 6. Chord distribution trong generated output
chord_map = {0:'None',1:'Major',2:'Minor',3:'Dom'}
chord_vals = np.round(all_gen[:,:,0].mean(axis=1)).astype(int).clip(0,3)
unique, counts_c = np.unique(y[:,3].astype(int), return_counts=True)
axes[1,2].pie(counts_c, labels=[chord_map[u] for u in unique],
              colors=['#95a5a6','#2ecc71','#e74c3c','#3498db'],
              autopct='%1.1f%%', startangle=90)
axes[1,2].set_title('Chord Distribution in Dataset')

plt.tight_layout()
plt.savefig('music_quality.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: music_quality.png')


## 13. Model Architecture Summary

In [ ]:
# Tong ket kien truc model
print('='*50)
print('GESTURHYTHM - MODEL ARCHITECTURE')
print('='*50)
print()
print('INPUT')
print(f'  Gesture sequence : (B, 30, 126)')
print(f'  30 frames x 126 features (2 hands x 21 landmarks x 3 coords)')
print()
print('ENCODER: Causal Transformer')
enc_params = sum(p.numel() for p in model.encoder.parameters())
print(f'  Linear embed     : 126 -> 128')
print(f'  Positional embed : 100 x 128')
print(f'  TransformerEncoder: 3 layers, 4 heads, ff=256')
print(f'  Causal mask      : YES (frame i cannot see frame j > i)')
print(f'  Parameters       : {enc_params:,}')
print()
print('DECODER: Autoregressive Music Decoder')
dec_params = sum(p.numel() for p in model.decoder.parameters())
print(f'  Note embed       : 3 -> 128')
print(f'  TransformerDecoder: 3 layers, 4 heads, cross-attention')
print(f'  Causal mask      : YES (note i cannot see note j > i)')
print(f'  Output           : 128 -> 3 (pitch, velocity, duration)')
print(f'  Parameters       : {dec_params:,}')
print()
print('OUTPUT')
print(f'  Note sequence    : (B, 16, 3)')
print(f'  16 notes x [pitch_norm, velocity_norm, duration_norm]')
print()
total_p = sum(p.numel() for p in model.parameters())
print(f'TOTAL PARAMETERS   : {total_p:,}')
print(f'FINAL VAL LOSS     : {val_losses[-1]:.4f}')
